# Notebook para crear los modelos para cada arquitectura

In [1]:
import pandas as pd
import numpy as np

Cargar los datos

In [2]:
datos_chicos=pd.read_csv("train_small.csv")
datos_grandes=pd.read_csv("train_big.csv")

# Glove

In [3]:
import gensim
import numpy as np
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
import nltk
nltk.download('punkt')
nltk.download('stopwords')

from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\angel\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\angel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
def load_glove_model(glove_file):
    print("Loading GloVe Model...")
    glove_model = {}
    with open(glove_file, 'r', encoding="utf-8") as f:
        for line in f:
            split_line = line.split()
            word = split_line[0]
            embedding = np.array([float(val) for val in split_line[1:]])
            glove_model[word] = embedding
    print("Done.", len(glove_model), "words loaded!")
    return glove_model

glove_file = '../glove_6B/glove-sbwc.i25.vec'
glove_model = load_glove_model(glove_file)

Loading GloVe Model...
Done. 855380 words loaded!


In [7]:
def tokenizar(data):
    respuestas = list(data['respuestas'].values)
    #counter=0 Este counter se utiliza si se desea limitar el número de oraciones en cada respuesta
    indice_extras=[]
    respuestas_token = []
    for i in respuestas:
        #Tokenize the text into sentences
        sentences = sent_tokenize(i)
        #if len(sentences)<3 or len(sentences)>100:
        #   indice_extras.append(counter)
        #else:
            # Tokenize each sentence into words
        tokenized_sentences = [word_tokenize(sentence) for sentence in sentences]
        respuestas_token.append(tokenized_sentences)
    
    stop_words = set(stopwords.words('spanish'))

    def sentence_embedding(sentence_tokens, embeddings, stop_words):
        embedding_dim = 300  # GloVe 50d
        sentence_vector = np.zeros(embedding_dim)
        word_count = 0
        
        for word in sentence_tokens:
            word = word.lower()
            if word not in stop_words and word in embeddings:
                sentence_vector += embeddings[word]
                word_count += 1
                
        if word_count > 0:
            sentence_vector /= word_count  # Optionally, normalize by number of words
        
        return sentence_vector
        # Calculate embeddings for each sentence, ignoring stopwords
    gloVe_Embedding=[]
    for tokenized_sentences in respuestas_token:

        sentence_embeddings = [sentence_embedding(sentence, glove_model, stop_words) for sentence in tokenized_sentences]
        sentence_embeddings = np.array(sentence_embeddings)
        gloVe_Embedding.append(sentence_embeddings)
    
    return gloVe_Embedding

In [8]:
embeddings_chicos=tokenizar(datos_chicos)
embeddings_grandes=tokenizar(datos_grandes)

# TDA

In [17]:
import ripser
from persim import plot_diagrams, PersistenceImager
import matplotlib.pyplot as plt
import gensim
import numpy as np
import pandas as pd

In [18]:
def generarImagenes(gloVe_Embedding): 
    lista_vectores = []
    for i in range(len(gloVe_Embedding)):
        ripserperiod = ripser.ripser(gloVe_Embedding[i])["dgms"]
        h0_diagram = ripserperiod[0].copy()
        h0_diagram = h0_diagram[np.isfinite(h0_diagram).all(axis=1)]
        lista_vectores.append(h0_diagram)

    #ignore warnings
    import warnings
    warnings.filterwarnings("ignore")

        # Generate the images for all the persistence diagrams at once to assure the same pixel size
    # Manually set birth and persistence ranges based on your data
    birth_range = (0, 0.5)  # Adjust these values as per your data
    pers_range = (0, 3)   # Adjust these values as per your data

    # Initialize PersistenceImager
    pimgr = PersistenceImager(pixel_size=0.01, birth_range=birth_range, pers_range=pers_range)
    pimgr.kernel_params = {'sigma': 0.01}
    pdgms = lista_vectores
    #pimgr.fit(lista_vectores, skew=True)
    pimgs = pimgr.transform(pdgms,skew=True)

    return pimgs

In [23]:
imagenes_chicos=generarImagenes(embeddings_chicos)
imagenes_grandes=generarImagenes(embeddings_grandes)

In [25]:
import pandas as pd
import numpy as np
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle

In [29]:
def generarmodeloLASSO(imagenes, labels, tamano=0):
    #Convert imagenes array from 3D to 2D
    lol=[]
    for i in range(len(imagenes)):
        lol.append(imagenes[i].flatten())
    imagenes=np.array(lol)
    
    X = imagenes#[:n_samples]
    y = labels#[:n_samples]

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Create a Logistic Regression model with L1 regularization
    alpha = 0.1
    # Regularization strength 0.001
    log_reg = LogisticRegression(penalty='l1', solver='liblinear', C=1/alpha)

    # Train the model
    log_reg.fit(X_train, y_train)

    # Save the model with pickle
    nombre_archivo = f'modeloLASSO_{tamano}.pkl'
    with open(nombre_archivo, 'wb') as f:
        pickle.dump(log_reg, f)


In [30]:
generarmodeloLASSO(imagenes_chicos, datos_chicos['ai'], tamano='chicos')
generarmodeloLASSO(imagenes_grandes, datos_grandes['ai'], tamano='grandes')

# LSTM

In [31]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Masking
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

In [37]:
def generarModeloLSTM(val_glove, labels, tamano=0):
    max_timesteps = 8  # Define the maximum length for padding

    X = val_glove
    y = labels

    # Pad the sequences to have the same length
    X_padded = pad_sequences(X, maxlen=max_timesteps, dtype='float32', padding='post', truncating='post')

    # Define the BiLSTM model
    model = Sequential()
    model.add(Masking(mask_value=0.0, input_shape=(max_timesteps, 300)))  # Mask padded values
    model.add(Bidirectional(LSTM(64, return_sequences=False)))
    model.add(Dense(1, activation='sigmoid'))

    # Compile the model
    model.compile(optimizer=Adam(), loss=BinaryCrossentropy(), metrics=['accuracy'])

    # Print the model summary
    model.summary()

    # Train the model
    model.fit(X_padded, y, epochs=5, batch_size=32, validation_split=0.2,verbose=0)

    # Save the model as a keras model
    nombre_archivo = f'modeloLSTM_{tamano}.keras'
    model.save(nombre_archivo)

In [38]:
generarModeloLSTM(embeddings_chicos, datos_chicos['ai'], tamano='chicos')
generarModeloLSTM(embeddings_grandes, datos_grandes['ai'], tamano='grandes')

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_4 (Masking)             │ (None, 8, 300)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 128)            │       186,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 187,009 (730.50 KB)

 Trainable params: 187,009 (730.50 KB)

 Non-trainable params: 0 (0.00 B)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_5 (Masking)             │ (None, 8, 300)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 128)            │       186,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 187,009 (730.50 KB)

 Trainable params: 187,009 (730.50 KB)

 Non-trainable params: 0 (0.00 B)

# BERT

In [3]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import tensorflow as tf

from tensorflow.keras import layers, models

import pandas as pd

In [4]:
def generarModeloBert(text, labels, tamano=0):

    text = text.tolist()  # Convert to list if it's a pandas Series
    labels = labels.tolist()  # Convert to list if it's a pandas Series

    model_name = "bert-base-uncased"
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

        # Tokenize
    encodings = tokenizer(text, truncation=True, padding=True, return_tensors="pt")

    # Create a dataset object
    class SimpleDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels
        
        def __getitem__(self, idx):
            item = {key: val[idx] for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx])
            return item
        
        def __len__(self):
            return len(self.labels)

    dataset = SimpleDataset(encodings, labels)

    # 3. Set up Trainer
    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        logging_dir="./logs",
        logging_steps=10,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
    )

    # 4. Train
    trainer.train()

    # Save the model
    nombre_archivo = f'modeloBERT_{tamano}'
    model.save_pretrained(nombre_archivo)
    tokenizer.save_pretrained(nombre_archivo)
    

In [5]:
generarModeloBert(datos_chicos['respuestas'].values, datos_chicos['ai'], tamano='chicos')
generarModeloBert(datos_grandes['respuestas'].values, datos_grandes['ai'], tamano='grandes')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
d:\anaconda3_5\envs\envtda\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.567400
20,0.526200
30,0.334200
40,0.216900


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
d:\anaconda3_5\envs\envtda\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.686900
20,0.521000
30,0.285200
40,0.094200
50,0.132800
60,0.156500
70,0.200600
80,0.232700
90,0.115400
100,0.293400


# Transformers


In [13]:

from tensorflow.keras import layers, models
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

In [14]:
def generarModeloTransformers(val_glove, labels, tamano=0):
    # Split data into training and validation sets (80% train, 20% validation)
    

    # Assume val_glove is a list/array of variable-length sequences of GloVe embeddings
        # Each embedding is a (embedding_dim,) vector

       # Assume val_glove is a list/array of variable-length sequences of GloVe embeddings
        # Each embedding is a (embedding_dim,) vector

        embedding_dim = 300
        max_timesteps = 9
        

        # Pad sequences to same length
        X_padded = pad_sequences(val_glove, maxlen=max_timesteps, dtype='float32', padding='post', truncating='post')

        # Create boolean mask: True where the sequence is non-zero (not padding)
        mask = (X_padded.sum(axis=-1) != 0)

        def expand_mask(m):
            return tf.cast(tf.expand_dims(tf.expand_dims(m, 1), 1), tf.float32)
    

        def create_transformer_binary_classifier(embedding_dim, num_heads, ff_dim):
            input_embeddings = tf.keras.Input(shape=(None, embedding_dim), name='input_embeddings')
            input_mask = tf.keras.Input(shape=(None,), dtype=tf.bool, name='input_mask')

                # Expand dims to shape (batch_size, 1, 1, seq_len) — required by MultiHeadAttention
            #expanded_mask = layers.Lambda(lambda m: tf.cast(tf.expand_dims(tf.expand_dims(m, 1), 1), tf.float32))(input_mask)
            expanded_mask = layers.Lambda(expand_mask)(input_mask)

            attention_output = layers.MultiHeadAttention(
                num_heads=num_heads,
                key_dim=embedding_dim,
                dropout=0.1
            )(input_embeddings, input_embeddings, attention_mask=expanded_mask)


            # Add & Norm
            x = layers.Add()([attention_output, input_embeddings])
            x = layers.LayerNormalization()(x)

            # Feed-forward network
            ffn_output = layers.Dense(ff_dim, activation='relu')(x)
            ffn_output = layers.Dense(embedding_dim)(ffn_output)

            # Add & Norm
            x = layers.Add()([ffn_output, x])
            x = layers.LayerNormalization()(x)

            # Global pooling (can replace with first-token approach if desired)
            pooled_output = layers.GlobalAveragePooling1D()(x)

            # Final classification head
            output = layers.Dense(1, activation='sigmoid')(pooled_output)

            model = models.Model(inputs=[input_embeddings, input_mask], outputs=output)
            model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
            return model
        model = create_transformer_binary_classifier(
        embedding_dim=embedding_dim,
        num_heads=4,
        ff_dim=128
        )

        model.summary()

        model.fit(
            {'input_embeddings': X_padded, 'input_mask': mask},
            labels,
            epochs=5,
            batch_size=16
        )
        # Save the model
        nombre_archivo = f'modeloTransformers_{tamano}.keras'
        model.save(nombre_archivo)



In [15]:
generarModeloTransformers(val_glove=embeddings_chicos, labels=datos_chicos['ai'], tamano='chicos')
generarModeloTransformers(embeddings_grandes, datos_grandes['ai'], tamano='grandes')

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_mask          │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_embeddings    │ (None, None, 300) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1, 1,      │          0 │ input_mask[0][0]  │
│                     │ None)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 300) │  1,443,900 │ input_embeddings… │
│ (MultiHeadAttentio… │                   │            │ input_embeddings… │
│                     │                   │            │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 300) │          0 │ multi_head_atten… │
│                     │                   │            │ input_embeddings… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, None, 300) │        600 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 128) │     38,528 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 300) │     38,700 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, None, 300) │          0 │ dense_1[0][0],    │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 300) │        600 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 300)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │        301 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,522,629 (5.81 MB)

 Trainable params: 1,522,629 (5.81 MB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_mask          │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_embeddings    │ (None, None, 300) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 1, 1,      │          0 │ input_mask[0][0]  │
│                     │ None)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 300) │  1,443,900 │ input_embeddings… │
│ (MultiHeadAttentio… │                   │            │ input_embeddings… │
│                     │                   │            │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, None, 300) │          0 │ multi_head_atten… │
│                     │                   │            │ input_embeddings… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 300) │        600 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, None, 128) │     38,528 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, None, 300) │     38,700 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, None, 300) │          0 │ dense_4[0][0],    │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 300) │        600 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 300)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │        301 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,522,629 (5.81 MB)

 Trainable params: 1,522,629 (5.81 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 9s 61ms/step - accuracy: 0.5754 - loss: 1.6708
Epoch 2/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.7282 - loss: 0.5146
Epoch 3/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.8810 - loss: 0.3461
Epoch 4/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.8667 - loss: 0.2770
Epoch 5/5
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.9599 - loss: 0.1636
